# Analysis of amounts

This notebook explores the different statistics that one can gather from extracts from Carrefour user account database. 

In [ ]:
import sys
sys.path.append("..")

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport src.carrefour_receipts_api.mongodb_querying
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt 
from carrefour_receipts_api.mongodb_querying import display_amounts


In [6]:
df_amounts = pd.read_csv('../data/20250501-carrefour_amounts.csv', parse_dates=['dateKey'])
df_loyalty = pd.read_csv('../data/20250501-carrefour_loyalty.csv', parse_dates=['date'])

In [7]:
df_amounts[df_amounts.recordType == 'receipt'][['dateKey', 'totalPaidAmount', 'vatTotalProductsAt20', 'vatTotalProductsAt5', 'vatAt5', 'vatAt20']]

,dateKey,totalPaidAmount,vatTotalProductsAt20,vatTotalProductsAt5,vatAt5,vatAt20
0,2024-10-31,4.18,4.18,0.00,0.00,0.70
1,2024-11-30,5.56,0.00,5.56,0.29,0.00
2,2022-04-25,78.69,8.59,68.35,3.55,1.43
3,2022-04-26,126.50,98.14,28.36,1.47,16.33
4,2022-05-03,79.56,18.90,60.66,3.15,3.15
...,...,...,...,...,...,...
272,2023-09-10,126.98,40.68,35.24,1.85,6.77
273,2023-09-17,16.98,0.00,0.00,0.00,0.00
274,2024-01-05,29.38,0.00,29.38,1.52,0.00
275,2024-01-26,4.44,2.49,1.95,0.10,0.41


In [21]:
df_amounts[df_amounts["vatTotalProductsAt5"] == 0]

,id,dateKey,recordType,totalAmountBeforeDiscount,couponDiscount,totalAmountImmediateDiscount,totalAmountDeferredDiscount,totalEarnedAmount,totalPaidAmount,vatTotalProductsAt20,...,CB,CBPASS,Cagnotte fidélité,Carte PASS,Carte bancaire,PAYPAL,Titre restaurant/service,eLOYALTY,payment.type.label.returned,totalTrueAmount
0,3020180195621_20241031_32-299-1553,2024-10-31,receipt,4.18,0.0,0.00,0.00,0.0,4.18,4.18,...,0.0,0.0,0.00,0.00,4.18,0.0,0.0,0.00,0.0,4.18000
2,3020339801200_20220425_6-227-1648,2022-04-25,receipt,79.81,0.0,-1.12,0.39,0.0,78.69,0.00,...,0.0,0.0,0.00,78.69,0.00,0.0,0.0,0.00,0.0,78.69000
4,3020339801200_20220503_1-9-1015,2022-05-03,receipt,79.56,0.0,0.00,4.85,0.0,79.56,0.00,...,0.0,0.0,0.00,79.56,0.00,0.0,0.0,0.00,0.0,79.56000
6,3020339801200_20220515_6-65-1129,2022-05-15,receipt,94.03,0.0,0.00,0.00,0.0,94.03,0.00,...,0.0,0.0,32.42,61.61,0.00,0.0,0.0,0.00,0.0,61.61000
7,3020339801200_20220520_6-166-1706,2022-05-20,receipt,89.82,0.0,-5.85,0.00,0.0,83.97,0.00,...,0.0,0.0,0.00,83.97,0.00,0.0,0.0,0.00,0.0,83.97000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,3020470210000_20230917_30-19-1042,2023-09-17,receipt,16.98,0.0,0.00,0.00,0.0,16.98,0.00,...,0.0,0.0,13.20,3.78,0.00,0.0,0.0,0.00,0.0,3.78000
275,3020470210000_20240126_20-101-1747,2024-01-26,receipt,4.44,0.0,0.00,0.00,0.0,4.44,0.00,...,0.0,0.0,4.44,0.00,0.00,0.0,0.0,0.00,0.0,0.00000
276,3020470210000_20240615_25-5-1530,2024-06-15,receipt,50.67,0.0,-20.83,0.00,0.0,29.84,0.00,...,0.0,0.0,23.15,6.69,0.00,0.0,0.0,0.00,0.0,6.69000
277,567287383,2024-02-29,order,111.14,0.0,-10.56,0.00,0.0,100.58,100.68,...,0.0,0.0,0.00,0.00,0.00,0.0,0.0,49.91,0.0,48.26385


In [114]:
# Group by 'date' and count unique IDs
filtered_dates = df_amounts.groupby('dateKey').filter(lambda group: group['id'].nunique() > 1)

In [71]:
filtered_dates_several = df_amounts.groupby('dateKey').filter(lambda group: (group['id'].nunique() > 1) and (group['totalTrueAmount'] > 0).sum() > 1).sort_values(by='dateKey')

In [111]:
filtered_dates_several[(filtered_dates_several.totalAmountDeferredDiscount == 0) & ((filtered_dates_several.totalTrueAmount > 0) & ((filtered_dates_several.dateKey.dt.year == 2024) | (filtered_dates_several.dateKey.dt.year == 2025)))]['dateKey']

181   2024-02-27
183   2024-02-29
277   2024-02-29
284   2024-03-08
192   2024-04-11
191   2024-04-11
196   2024-04-30
195   2024-04-30
281   2024-05-17
278   2024-05-24
208   2024-06-27
288   2024-06-27
226   2024-09-20
289   2024-09-20
228   2024-09-28
229   2024-09-28
293   2024-10-19
234   2024-10-19
267   2025-04-17
Name: dateKey, dtype: datetime64[ns]

In [115]:
df_loyalty.groupby(by="date").count()

,operationId,earned,burned
date,,,
2024-05-04,1,1,1
2024-05-06,1,1,1
2024-05-09,1,1,1
2024-05-10,1,1,1
2024-05-17,2,2,2
...,...,...,...
2025-04-05,3,3,3
2025-04-12,1,1,1
2025-04-17,2,2,2


In [47]:
filtered_dates.dateKey.nunique()

38

In [5]:
grouped_loyalty = (
    df_amounts[
        (df_amounts["dateKey"].dt.year == 2024)
        | (df_amounts["dateKey"].dt.year == 2025)
    ]
    .groupby(by=[df_amounts.dateKey.dt.year, df_amounts.dateKey.dt.month])
    .sum(numeric_only=True)
)

# grouped_loyalty_all = grouped_loyalty.join(other=df_loyalty.groupby(
#     by=[df_loyalty["date"].dt.year, df_loyalty["date"].dt.month]
# ).sum(numeric_only=True), on=['date', 'date'], how='inner')

In [9]:
grouped_loyalty[['totalPaidAmount', 'Bons de réduction','Cagnotte fidélité', 'eLOYALTY', 'totalTrueAmount']]

totalPaidAmount  Bons de réduction  Cagnotte fidélité  \
dateKey dateKey                                                          
2024    1                 542.41               4.40              82.62   
        2                 822.14               2.00             132.53   
        3                 630.60               0.50               9.94   
        4                 737.27               0.00              40.89   
        5                 514.16               0.00              46.97   
        6                 363.48               0.00              27.52   
        7                 421.99               0.00              14.35   
        8                 718.58               0.00               0.00   
        9                 861.95               3.50              22.76   
        10                423.80               0.00              60.70   
        11                469.45               1.95              20.19   
        12                667.33               0.00              26.36   
2025    1                 580.51               0.00             145.78   
        2                 503.61               0.00              38.03   
        3                 484.42               0.00              63.96   
        4                 641.65               0.80              65.34   

                 eLOYALTY  totalTrueAmount  
dateKey dateKey                             
2024    1            0.00        437.86385  
        2           49.91        618.22220  
        3           12.00        584.04945  
        4            2.11        689.62825  
        5            0.00        451.23525  
        6            0.98        326.39805  
        7            9.24        388.50000  
        8           12.82        688.66000  
        9           31.51        775.04970  
        10           4.00        347.25465  
        11           0.00        435.81835  
        12           0.00        620.90675  
2025    1           15.80        402.78310  
        2            0.00        451.12870  
        3            0.00        407.29030  
        4            0.00        554.05265

# Dashboard

KPIs to define.

In [ ]:
import streamlit as st
import seaborn as sns

st.title("Carrefour Receipts KPI Dashboard")

# Line chart
st.subheader("KPI Trends Over Time")
st.line_chart(df.set_index("date")[["sales_volume", "avg_receipt_amount"]])

# Heatmap
st.subheader("Data Consumption Heatmap")
st.pyplot(sns.heatmap(heatmap_data, cmap="YlGnBu").figure)

In [ ]:
# use Dash as a microservice for interactive visualizations
from dash import Dash, dcc, html, Input, Output
import pandas as pd
import plotly.express as px

# Load data
df = pd.read_csv("data.csv")

# Initialize app
app = Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1("Carrefour Receipts Dashboard"),
    dcc.RangeSlider(
        id="date-range",
        min=2020,
        max=2023,
        step=1,
        marks={i: str(i) for i in range(2020, 2024)},
        value=[2020, 2023]
    ),
    dcc.Graph(id="sales-graph")
])

# Callback
@app.callback(
    Output("sales-graph", "figure"),
    Input("date-range", "value")
)
def update_graph(date_range):
    filtered_df = df[(df["year"] >= date_range[0]) & (df["year"] <= date_range[1])]
    fig = px.line(filtered_df, x="date", y="sales_volume", title="Sales Volume Over Time")
    return fig

# Run app
if __name__ == "__main__":
    app.run_server(debug=True)